# ASTRA-SHIELD ML Experiments Notebook

**Purpose**: Develop, test, and validate disaster detection models

**Author**: Person 1 - ML/AI Engineer

---

## 1. Environment Setup

Test if all packages are installed correctly.

In [ ]:
import sys
import torch
import numpy as np
import cv2
import pandas as pd
from pathlib import Path

print("=" * 60)
print("ASTRA-SHIELD ML ENVIRONMENT CHECK")
print("=" * 60)
print(f"✅ Python: {sys.version}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
print(f"✅ Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"✅ NumPy: {np.__version__}")
print(f"✅ OpenCV: {cv2.__version__}")
print("=" * 60)

## 2. Load and Test Preprocessing

Load satellite images and apply preprocessing pipeline.

In [ ]:
# Add parent directory to path
sys.path.append('..')

from preprocessing.preprocess import SatelliteImagePreprocessor, DataAugmenter
import matplotlib.pyplot as plt

# Initialize preprocessor
preprocessor = SatelliteImagePreprocessor(img_size=512, normalize=True)

print("✅ Preprocessor initialized")
print(f"  - Target size: 512x512")
print(f"  - Normalization: Enabled")

# Test on demo image if available
demo_dir = Path("../demo/flood")
if demo_dir.exists():
    image_files = list(demo_dir.glob("*.png")) + list(demo_dir.glob("*.tif"))
    if image_files:
        print(f"\n📷 Found {len(image_files)} demo images")

## 3. Load Classification Model

Test the DisasterClassifier with ResNet50.

In [ ]:
from models.classifier.disaster_classifier import DisasterClassifier

classifier = DisasterClassifier(device='cpu', pretrained=True)

print("✅ DisasterClassifier loaded")
print(f"  Classes: {classifier.CLASSES}")

# Test prediction
dummy_image = torch.randn(1, 3, 512, 512)
result = classifier.predict(dummy_image)

print(f"\n🎯 Test Prediction:")
print(f"  Disaster Type: {result['disaster_type']}")
print(f"  Confidence: {result['confidence']:.2%}")
print(f"  Predictions: {result['predictions']}")

## 4. Load Segmentation Model

Test the DisasterSegmenter with U-Net.

In [ ]:
from models.segmentation.disaster_segmenter import DisasterSegmenter

segmenter = DisasterSegmenter(device='cpu')

print("✅ DisasterSegmenter loaded")

# Test prediction
mask, confidence = segmenter.predict(dummy_image, threshold=0.5)

print(f"\n🎯 Test Segmentation:")
print(f"  Mask shape: {mask.shape}")
print(f"  Affected pixels: {mask.sum()}")
print(f"  Confidence: {confidence:.2%}")

# Calculate metrics
area = segmenter.calculate_affected_area(mask)
severity = segmenter.calculate_severity(mask)

print(f"  Affected area: {area:.2f} km²")
print(f"  Severity: {severity:.1f}/10")

## 5. Test Complete Pipeline

Run the full DisasterAnalyzer end-to-end.

In [ ]:
from inference.predict import DisasterAnalyzer

# Initialize
analyzer = DisasterAnalyzer(device='cpu')

print("✅ DisasterAnalyzer initialized")
print("\nPipeline components:")
print("  ✅ Preprocessor")
print("  ✅ Classifier")
print("  ✅ Segmenter")
print("  ✅ Metrics")

## 6. Evaluation Metrics

Understand IoU and Dice coefficient for segmentation.

In [ ]:
from evaluation.metrics import SegmentationMetrics, ClassificationMetrics

# Example: Calculate metrics on random masks
pred_mask = np.random.rand(512, 512) > 0.5
true_mask = np.random.rand(512, 512) > 0.5

iou = SegmentationMetrics.iou(pred_mask, true_mask)
dice = SegmentationMetrics.dice(pred_mask, true_mask)
acc = SegmentationMetrics.pixel_accuracy(pred_mask, true_mask)

print("📊 Example Segmentation Metrics:")
print(f"  IoU (Intersection over Union): {iou:.3f}")
print(f"  Dice Coefficient: {dice:.3f}")
print(f"  Pixel Accuracy: {acc:.3f}")

print("\n💡 Metric Interpretation:")
print("  IoU > 0.70 → Good segmentation")
print("  Dice > 0.75 → Good segmentation")
print("  Accuracy > 0.90 → Good pixel-level classification")

## 7. Next Steps

### If you have labeled data:
1. Store in `ml/datasets/flood/`, `ml/datasets/wildfire/`, etc.
2. Run `preprocessing/preprocess.py` to load and augment
3. Fine-tune models on your specific disaster types
4. Evaluate performance using `evaluation/evaluate.py`

### If you don't have labeled data:
1. Use transfer learning (ImageNet weights)
2. Test on public datasets (Kaggle, NASA, USGS)
3. Document baseline performance

### To integrate with backend/frontend:
1. Person 2: Import `from ml.inference.predict import api_analyze_image`
2. Person 3: Call backend API endpoint `/api/analyze`
3. See results in web dashboard

---

**🚀 ML Pipeline is ready for development!**